# L10 · Alfabetos no latinos

| | |
|---|---|
| **Qué vas a aprender** | Por qué `"Ibrahim"` y `"ıbrahim"` no son la misma persona para Python |
| **Cuánto tiempo** | 90 min |
| **Datasets** | `clientes_crm.csv` |

> **Al terminar vas a poder** decidir, con números, si tu pipeline de normalización
> está uniendo personas o fusionando personas distintas.

In [ ]:
import sys
from pathlib import Path

# JupyterLab arrancado en la raíz del proyecto deja el cwd ahí.
# Si abres el notebook con el cwd en notebooks/, sube un nivel.
ROOT = Path.cwd()
if not (ROOT / "datos").exists() and (ROOT.parent / "datos").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks"))
DATOS = ROOT / "datos"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from herramientas import (cargar, mostrar, ficha, guardar_csv, guardar_excel,
                           guardar_figura, parsear_importes, parsear_fecha,
                           clave_canonica, detectar_alfabeto, auditar_fusiones,
                           resumen_entrega, perfil_memoria, contar_falsos_nulos)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
plt.style.use("seaborn-v0_8-whitegrid")
pd.__version__

In [ ]:
clientes = pd.read_csv(DATOS / "clientes_crm.csv", encoding="utf-8-sig")
print("filas:", f"{len(clientes):,}")
mostrar(clientes["nombre"].drop_duplicates().head(20), 20, "20 NOMBRES (columna 1)")

## El concepto

Unicode asigna un número a cada carácter. `"I"` y `"ı"` son dos números distintos, y
Python los trata como dos cadenas distintas. Ni es un bug ni es raro: en turco son dos
caracteres reales que se pronuncian igual.

Lo que **no** es raro es lo que hace tu código cuando los compara: une a unos clientes
y separa a otros, sin avisar.

## Demostración 1: el turco I / ı / İ

In [ ]:
import unicodedata as ud

for ch in ["I", "İ", "ı", "i"]:
    print(f"{ch!r:6} {ud.name(ch):40} U+{ord(ch):04X}  "
          f"lower={ch.lower()!r:7} (len {len(ch.lower())})  "
          f"casefold={ch.casefold()!r:7} (len {len(ch.casefold())})")

In [ ]:
# el resultado que rompe joins:
print("Ibrahim == ıbrahim ->", "Ibrahim" == "ıbrahim")
print("İbrahim == Ibrahim ->", "İbrahim" == "Ibrahim")
print()
# ojo: casefold puede devolver MAS de un caracter
n_orig, n_cf = len("İ"), len("İ".casefold())
print(f"len('İ') = {n_orig}  ->  len(casefold) = {n_cf}",
      "  <-- 1 caracter se ha convertido en 2")

**Por qué `str.lower()` no arregla el turco.** El `lower()` de Python es
*locale-independent*: `'I'.lower()` da `'i'`, que es lo correcto según el estándar
Unicode, pero es **incorrecto** para el turco, donde la `I` minúscula es `ı`. No hay
bandera que lo cambie: la solución no es "arreglar `lower`", es generar una **clave
canónica** que trate esos caracteres a mano.

## Demostración 2: `casefold` ALARGA la cadena

In [ ]:
print(f"{'entrada':8} {'len':>4} {'lower':>8} {'len':>4} {'casefold':>9} {'len':>4}")
print("-" * 46)
for ch in ["ß", "ẞ", "ﬀ"]:
    print(f"{ch!r:8} {len(ch):>4} {ch.lower()!r:>8} {len(ch.lower()):>4} "
          f"{ch.casefold()!r:>9} {len(ch.casefold()):>4}")
print()
print('Una regla que asuma "1 letra = 1 caracter" ya es falsa en Unicode.')

## Demostración 3: caracteres que se parecen y NO son iguales

In [ ]:
familia = [
    ("s", "U+0073", "s basica"),
    ("ş", "U+015F", "s con CEDILLA (turco)"),
    ("ș", "U+0219", "s con COMBA   (rumano)"),
    ("ś", "U+015B", "s con ACENTO  (polaco)"),
]

for ch, cp, desc in familia:
    print(f"  {ch}  {cp}  mayuscula={ch.upper()!r:5} {desc}")

print()
print("son 4 caracteres distintos:", len({c for c, _, _ in familia}) == 4)
print("pero se ven iguales. En un catalogo con ruido, se cuelan solos.")

Fíjate en el detalle: `ş.upper()` es `Ş` y `ș.upper()` es `Ș`. Son dos
caracteres distintos con un trazo casi idéntico. Un catálogo de productos lleno de
`ş` y `ș` mezclados no se puede limpiar con `lower()`.

## Demostración 4: el árabe

In [ ]:
arabe = ["أحمد", "أَحْمَد", "احمد"]

for s in arabe:
    print(f"  {s!r:16} len={len(s)}  repr={s!r}")

print()
print("Las dos primeras se VEN IGUALES pero tienen distinta longitud:",
      len(arabe[0]), "!=", len(arabe[1]))
print("Motivo: el tashkil (marcas U+064B-0652) son caracteres invisibles.")
print("Y la direccion RTL usa marcas U+200E / U+200F que tampoco se ven.")
print()
print("CONSECUENCIA: nunca valides texto arabe con len(), ni compares longitudes.")

In [ ]:
clientes["script"] = detectar_alfabeto(clientes["nombre"])
clientes["clave"] = clave_canonica(clientes["nombre"])

print(clientes["script"].value_counts().to_string())
print()
print("nombres distintos  :", clientes["nombre"].nunique())
print("claves canonicas   :", clientes["clave"].nunique())
print("claves vacias      :", (clientes["clave"].str.strip() == "").sum(),
      "  <-- si esto es > 0, tu pipeline ESTA destruyendo texto")

## La solución: una clave canónica en capas

In [ ]:
import inspect
from herramientas import transliterar_latin
print(inspect.getsource(transliterar_latin))

El orden importa, y equivocarse cambia el resultado. Lo que hace
`transliterar_latin`, en este orden:

1. **Resuelve ligaduras y tes** (`ﬀ` → `ff`) que NFKD no separa.
2. **Quita tashkil y marcas de dirección** del árabe (U+064B-0652, U+200E/F). No se
   transliteran: se borran, porque son marcas, no letras.
3. **Translitera el latín extendido** con una tabla explícita: `ı`→`i`, `İ`→`I`,
   `ł`→`l`, `ø`→`o`, `ß`→`ss`. `ü` pasa a `u`, no a `ue`.
4. **NFKD y quita diacríticos**: `é` → `e`. El hamza árabe también cae aquí,
   porque NFKD lo separa y es una marca combinante.

Y una decisión de diseño importante: **los caracteres no latinos se quedan**. Aquí es
donde la mayoría de pipelines meta la pata. `unidecode("محمد")` devuelve `"mhmd"`, que
no es el nombre: es una aproximación inventada. Si aplicas eso a toda la columna,
`أحمد` y `Ahmad` se vuelven el mismo cliente.

In [ ]:
# ¿Qué le pasa a un pipeline ingenuo que solo deja [a-z0-9]?
ingenuo = (clientes["nombre"].str.casefold()
           .str.replace(r"[^a-z0-9]+", " ", regex=True).str.strip())

arabes = clientes[clientes["script"] == "arabic"]
print("  filas arabes            :", f"{len(arabes):,}")
print("  nombres arabes distintos:", arabes["nombre"].nunique())
print("  clave ingenua distinta  :", ingenuo[arabes.index].nunique(),
      "  <-- si esto es 1 o 2, has FUSIONADO clientes arabes distintos")
print("  clave correcta distinta :", arabes["clave"].nunique())

## ⚠️ La trampa

Tu clave canónica **funde** a personas distintas. En ASCII no hay un símbolo que
distinga `Ł` de `L`, ni `ş` de `ș`.

Y una clave no puede aplicar dos reglas alemanas a la vez. `ö` → `o` junta
`Söderberg` con `Soderberg` y separa `Müller` de `Mueller`. `ö` → `oe` haría lo
contrario. Hay que elegir y medir el coste.

In [ ]:
pares = [("Söderberg", "Soderberg"), ("Müller", "Mueller"),
        ("Łukasz", "Lukasz"), ("İbrahim", "Ibrahim")]
for a, b in pares:
    ka = clave_canonica(pd.Series([a])).iloc[0]
    kb = clave_canonica(pd.Series([b])).iloc[0]
    print(f"  {a:12} -> {ka:12} | {b:12} -> {kb:12} | igual={ka == kb}")

In [ ]:
auditar_fusiones(clientes, "nombre", clave=clientes["clave"], n=10)

In [ ]:
# quantify el coste: de las unificaciones, cuantas unen al MISMO cliente
# escrito de otra forma, y cuantas a gente DISTINTA?
grupos = clientes.groupby("clave")["nombre"].nunique()
colapsados = int((grupos - 1).sum())

print(f"grupos distintos tras normalizar : {len(grupos):,}")
print(f"variantes unificadas             : {colapsados:,}")
print()
print("La pregunta que importa: de esas", colapsados, "unificaciones,")
print("¿cuantas unen al MISMO cliente y cuantas a gente DISTINTA?")
print("Se responde cruzando la clave con 'pais' y con 'email' -> EJERCICIO 3")

## 🧪 Tu turno

- **Ejercicio 1** — ¿Cuántas filas tienen algún carácter árabe? Comprueba que tu
  cuenta coincide con `(clientes["script"] == "arabic").sum()`.
- **Ejercicio 2** — Detecta tashkil: ¿cuántos nombres contienen marcas U+064B–U+0652?
  `أَحْمَد` está sembrado en el CRM; `Muḥammad` no cuenta, la `ḥ` es latina.
- **Ejercicio 3** — Tasa de falsos positivos: grupos de `clave` con más de un
  `cliente_id`, divididos entre los grupos que han colapsado al menos dos filas.
  El email se conserva en las variantes del mismo cliente, así que un grupo con
  varios emails también es sospechoso. El país no: las variantes no lo cambian.
- **Ejercicio 4** — Para qué sirve `unidecode` y para qué no. Una frase para cada.

## 📦 Workbook integrador: normalización multilingüe

**Contexto.** Tienes una empresa con operaciones en España, Turquía, EAU, Alemania,
Suecia y Polonia. El CRM de cada país exporta su propia ortografía, y al importarlos
todos al mismo sistema tienes **el mismo cliente 2-3 veces**. El equipo de ventas está
mandando tres veces la misma propuesta a la misma persona, y tu jefa quiere arreglarlo.

**Tu trabajo.**

1. Construye la `clave_canonica` con `herramientas.clave_canonica`.
2. Quédate con un registro por clave. Eso es **un nombre normalizado**, no un
   cliente: dos `José García` distintos comparten clave.
3. **Mide el coste**: exporta a `salida/L10_fusiones_sospechosas.csv` los grupos
   donde la clave junta más de un `cliente_id`.
4. Escribe cuántas claves tienes y la tasa de falsos positivos
   (grupos con varios `cliente_id` / grupos que colapsan al menos dos filas).

In [ ]:
clientes["clave"] = clave_canonica(clientes["nombre"])
# 'alta' es texto dd/mm/yyyy. Ordenarla como texto no es ordenarla en el tiempo:
# '01/06/2024' queda delante de '20/01/2021'.
clientes["alta_dt"] = pd.to_datetime(clientes["alta"], format="%d/%m/%Y")

unicos = (clientes.sort_values("alta_dt")
                .drop_duplicates(subset="clave", keep="first")
                .reset_index(drop=True))

print(f"filas originales : {len(clientes):,}")
print(f"claves           : {len(unicos):,}")
print(f"cliente_id distintos: {clientes['cliente_id'].nunique():,}")
print(f"filas colapsadas : {len(clientes) - len(unicos):,}")

In [ ]:
g = clientes.groupby("clave").agg(
    ids=("cliente_id", "nunique"),
    emails=("email", "nunique"),
    nombres=("nombre", lambda s: sorted(set(map(str, s)))[:5]),
    filas=("nombre", "size"),
)

grupos_colapsados = g[g["filas"] > 1]
sospechosos = g[g["ids"] > 1].sort_values(["ids", "filas"], ascending=False)
tasa = len(sospechosos) / len(grupos_colapsados) if len(grupos_colapsados) else 0

# El mismo cliente_id partido en varias claves: la clave no ha reunido
# una variante que el generador sí sembró.
partidos = int((clientes.groupby("cliente_id")["clave"].nunique() > 1).sum())

print(f"grupos que colapsan filas     : {len(grupos_colapsados):,}")
print(f"grupos con varios cliente_id  : {len(sospechosos):,}")
print(f"tasa de falsos positivos      : {tasa:.1%}")
print(f"cliente_id partidos en 2 claves: {partidos:,}")
print()
mostrar(sospechosos.head(10), 10, "FUSIONES SOSPECHOSAS (revisar a mano)")

In [ ]:
reporte = sospechosos.reset_index()
reporte["variantes"] = reporte["nombres"].map(lambda vs: " | ".join(vs))

ruta_fusiones = guardar_csv(
    reporte[["clave", "variantes", "ids", "emails", "filas"]],
    "L10_fusiones_sospechosas",
)
ruta_unicos = guardar_csv(
    unicos.drop(columns="alta_dt"), "L10_clientes_unicos"
)
resumen_entrega(ruta_fusiones, ruta_unicos)

### Criterios de aceptación

1. `unicos` tiene exactamente `clientes["clave"].nunique()` filas.
2. `unicos["clave"].is_unique` es `True`.
3. `(unicos["clave"].str.strip() == "").sum()` es **0**.
4. `salida/L10_fusiones_sospechosas.csv` lista los grupos con más de un
   `cliente_id`, ordenados de más ids a menos.
5. La tasa de falsos positivos es `len(sospechosos) / len(grupos_colapsados)`.
   `partidos` (el mismo `cliente_id` en dos claves) tiene que salir 0: las
   variantes sembradas comparten email y la clave las reúne.